In [ ]:
# Cell 1: Set MUJOCO_GL before any imports (required for headless EGL rendering)
import os
os.environ['MUJOCO_GL'] = 'egl'

import stable_worldmodel as swm

world = swm.World('swm/OGBScene-v0', num_envs=4, image_shape=(96, 96), max_episode_steps=50)
print('Action space:', world.envs.single_action_space)

In [ ]:
# Cell 2: Collect episodes with random policy
random_policy = swm.policy.RandomPolicy(seed=42)
world.set_policy(random_policy)

world.collect(
    'data/ogbscene_demo.lance',
    episodes=20,
    seed=0,
    mode='overwrite',
)
print('Done collecting!')

In [ ]:
# Cell 3: Load dataset
abs_path = os.path.abspath('data/ogbscene_demo.lance')
dataset = swm.data.load_dataset(
    abs_path,
    frameskip=1,
    num_steps=50,
    keys_to_load=['pixels'],
)
print(f'Number of episodes: {len(dataset)}')
print(f'Pixels shape per episode: {dataset[0]["pixels"].shape}')  # (50, 3, 96, 96)

In [ ]:
# Cell 4: Visualize one episode — show 8 evenly-spaced frames
import matplotlib.pyplot as plt
import numpy as np

sample = dataset[0]
pixels = sample['pixels']  # (T, 3, H, W)

num_frames = 8
indices = np.linspace(0, pixels.shape[0] - 1, num_frames, dtype=int)

fig, axes = plt.subplots(1, num_frames, figsize=(20, 3))
for ax, idx in zip(axes, indices):
    img = pixels[idx].permute(1, 2, 0).numpy().astype(np.uint8)
    ax.imshow(img)
    ax.set_title(f't={idx}')
    ax.axis('off')

plt.suptitle('OGBScene — random policy trajectory', y=1.02)
plt.tight_layout()
plt.savefig('/workspace/swm-runpod/ogbscene_trajectory.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved to ogbscene_trajectory.png')

In [ ]:
# Cell 5 (optional): Show first frame of several episodes side-by-side
# to see the env's visual variation across episodes
n_episodes = min(8, len(dataset))

fig, axes = plt.subplots(1, n_episodes, figsize=(20, 3))
for i, ax in enumerate(axes):
    img = dataset[i]['pixels'][0].permute(1, 2, 0).numpy().astype(np.uint8)
    ax.imshow(img)
    ax.set_title(f'ep {i}')
    ax.axis('off')

plt.suptitle('OGBScene — initial frame per episode (visual variation)', y=1.02)
plt.tight_layout()
plt.savefig('/workspace/swm-runpod/ogbscene_variation.png', dpi=100, bbox_inches='tight')
plt.show()